# 2 — Transform

This notebook covers:
1. **Data cleaning** — null checks, duplicate checks, corrupt-file exclusion
2. **Split assignment** — pull train/val/test labels from the database
3. **Feature engineering** — scaling, encoding (to be added)

## 1. Data Cleaning

In [1]:
import os
import pandas as pd
import psycopg2

df = pd.read_csv('../data/features_3_sec.csv')
print(f'Shape: {df.shape}')  # expect (9990, 60)

ModuleNotFoundError: No module named 'psycopg2'

### Null check

No missing values are expected — the CSVs were pre-extracted from raw audio with no gaps.

In [ ]:
null_count = df.isnull().sum().sum()
print(f'Null values: {null_count}')  # expect 0

### Duplicate check

In [ ]:
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')  # expect 0

### Exclude corrupt file — `jazz.00054`

`jazz.00054.wav` fails to load (corrupted raw audio). Its 10 clips appear in the CSV because features were pre-extracted before the issue was discovered. We drop all rows whose parent song is `jazz.00054` before any further processing.

In [ ]:
df['parent'] = df['filename'].str.replace(r'\.wav$', '', regex=True).str.rsplit('.', n=1).str[0]

before = len(df)
df = df[df['parent'] != 'jazz.00054'].reset_index(drop=True)
after = len(df)

print(f'Dropped {before - after} rows (jazz.00054 clips)')
print(f'Remaining rows: {after}')  # expect 9980

## 2. Split Assignment

Splits are stored in the `audio_clips` table in our local PostgreSQL database (see `db/populate.py`). We pull them here and merge onto the dataframe.

**Why parent-song-level splits?**  
Each 30-second song is cut into ten 3-second clips. If clip 0 goes to train and clip 5 goes to test, the model can memorize the song's acoustic fingerprint across folds — that's data leakage. By assigning all clips from the same song to the same fold, we guarantee that nothing the model sees at train time appears in val or test.

In [ ]:
db_url = os.environ['DATABASE_URL']
conn = psycopg2.connect(db_url)

splits = pd.read_sql('SELECT file_path, split FROM audio_clips', conn)
conn.close()

df = df.merge(splits, left_on='filename', right_on='file_path', how='inner')
print(f'Rows after join: {len(df)}')  # expect 9980

### Split distribution

In [ ]:
distribution = (
    df.groupby(['split', 'label'])
    .size()
    .unstack(fill_value=0)
    .loc[['train', 'val', 'test']]
)
distribution['total'] = distribution.sum(axis=1)
distribution